# ◆ Análise Silver — Dados Limpos e Padronizados
**Pipeline Metrópole SP · Arquitetura Medallion**

> A camada Silver contém os dados após **limpeza, padronização e validação**.
> Este notebook compara Bronze vs Silver e analisa a qualidade resultante.


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#070b14', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1a3050',   'grid.color': '#1a3050',
    'text.color': '#d0e4ff',       'axes.labelcolor': '#7a9ab8',
    'xtick.color': '#4a6a8a',      'ytick.color': '#4a6a8a',
    'axes.titlecolor': '#00d4ff',  'axes.titlesize': 13,
    'axes.titleweight': 'bold',    'axes.grid': True,
    'figure.dpi': 120,
})
CYAN, GREEN, PURPLE = '#00d4ff', '#00ff88', '#7b2fff'
ORANGE, PINK, GOLD  = '#ff6b00', '#ff2d78', '#ffd700'
NEON = [CYAN, GREEN, PURPLE, ORANGE, PINK, GOLD]

# Detecta raiz do projeto
BASE = Path(os.environ.get('METRO_SP_BASE', Path.cwd()))
if not (BASE / 'data').exists():
    BASE = BASE.parent
BRONZE = BASE / 'data' / 'bronze'
SILVER = BASE / 'data' / 'silver'
GOLD   = BASE / 'data' / 'gold'
print(f"📂 Projeto: {BASE}")
print(f"   Bronze:  {BRONZE.exists()} | Silver: {SILVER.exists()} | Gold: {GOLD.exists()}")


In [ ]:
# ─── Carrega Bronze e Silver para comparação ─────────────────────────────────
br_ar   = pd.read_parquet(BRONZE / 'air_quality_raw.parquet')
sv_ar   = pd.read_parquet(SILVER / 'air_quality_clean.parquet')
br_gps  = pd.read_parquet(BRONZE / 'gps_bus_raw.parquet')
sv_gps  = pd.read_parquet(SILVER / 'gps_bus_clean.parquet')
br_tr   = pd.read_parquet(BRONZE / 'iot_traffic_raw.parquet')
sv_tr   = pd.read_parquet(SILVER / 'iot_traffic_clean.parquet')
br_ouv  = pd.read_parquet(BRONZE / 'ouvidoria_cdc_raw.parquet')
sv_ouv  = pd.read_parquet(SILVER / 'ouvidoria_clean.parquet')

print("  Fonte               Bronze     Silver    Retenção")
print("  " + "-"*50)
for nome, b, s in [
    ('Qualidade do Ar', br_ar, sv_ar),
    ('GPS Ônibus',      br_gps, sv_gps),
    ('IoT Tráfego',    br_tr, sv_tr),
    ('Ouvidoria',       br_ouv, sv_ouv),
]:
    ret = s is not None and b is not None
    n_b = len(b); n_s = len(s)
    print(f"  {nome:<20} {n_b:>6,}  →  {n_s:>6,}   {n_s/n_b:.1%}")


In [ ]:
# ─── Qualidade do Ar: Antes × Depois ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8), facecolor='#070b14')
fig.suptitle('Qualidade do Ar — Bronze vs Silver', color=CYAN, fontweight='bold', fontsize=14)

poluentes = ['mp10', 'mp25', 'o3', 'no2', 'co', 'so2']
for ax, pol in zip(axes.flat, poluentes):
    # Bronze: tenta converter (chega como string/object)
    b_vals = pd.to_numeric(br_ar[pol].replace('', np.nan), errors='coerce').dropna()
    s_vals = sv_ar[pol].dropna()

    ax.hist(b_vals, bins=30, alpha=0.5, color=ORANGE, label='Bronze', density=True)
    ax.hist(s_vals, bins=30, alpha=0.7, color=CYAN,   label='Silver', density=True)
    ax.set_title(pol.upper(), fontsize=10)
    ax.legend(fontsize=7)
    # Marca negativos
    n_neg_b = (b_vals < 0).sum()
    if n_neg_b > 0:
        ax.axvline(0, color=PINK, linestyle='--', linewidth=1.2,
                   label=f'{n_neg_b} negativos removidos')
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()
print("\nTransformações aplicadas na Silver:")
print("  ✅ Strings vazias → NaN")
print("  ✅ Datas DD/MM/YYYY → timestamp_utc (UTC+0)")
print("  ✅ CAC001 CO: ppm × 1.165 → µg/m³")
print("  ✅ MP2.5 negativos → removidos")
print("  ✅ IQAr calculado pela metodologia CETESB")


In [ ]:
# ─── IQAr: Série Temporal e Distribuição ─────────────────────────────────────
sv_ar['hora'] = pd.to_datetime(sv_ar['timestamp_utc']).dt.floor('h')
agg = sv_ar.groupby(['hora', 'estacao_id'])['iqar'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5), facecolor='#070b14')
fig.suptitle('IQAr — Camada Silver (Análise)', color=CYAN, fontweight='bold')

estacoes = agg['estacao_id'].unique()
for i, est in enumerate(estacoes):
    d = agg[agg.estacao_id == est]
    axes[0].plot(d['hora'], d['iqar'], label=est, color=NEON[i % len(NEON)],
                 linewidth=1.5, alpha=0.85)
axes[0].axhline(80, color=ORANGE, linestyle='--', linewidth=1.2, label='Limite RUIM')
axes[0].axhline(40, color=GREEN,  linestyle='--', linewidth=1.0, label='Limite MODERADO')
axes[0].set_title('IQAr por Estação — Série Temporal')
axes[0].legend(fontsize=8, loc='upper right')
axes[0].set_ylabel('IQAr')

axes[1].hist(sv_ar['iqar'].dropna(), bins=40, color=PURPLE, alpha=0.85, edgecolor='none')
axes[1].axvline(sv_ar['iqar'].mean(), color=CYAN,  linestyle='--', linewidth=2,
                label=f"Média = {sv_ar['iqar'].mean():.1f}")
axes[1].axvline(80, color=ORANGE, linestyle='--', linewidth=1.5, label='Limite RUIM')
axes[1].set_title('Distribuição do IQAr')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
n_ruim = (sv_ar['iqar'] > 80).sum()
print(f"\n  Registros com IQAr RUIM ou pior: {n_ruim} ({n_ruim/len(sv_ar):.1%})")


In [ ]:
# ─── GPS: Lotação após Padronização ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#070b14')
fig.suptitle('GPS Ônibus — Silver: Lotação e Posição', color=CYAN, fontweight='bold')

CLR_LOT = {'VAZIA': GREEN, 'MEIA': '#8bc34a', 'CHEIA': GOLD, 'LOTADA': PINK,
           'DESCONHECIDA': '#607090'}

# Bronze: enum caótico
axes[0].barh(br_gps['lotacao'].value_counts().index,
             br_gps['lotacao'].value_counts().values,
             color=ORANGE, alpha=0.75)
axes[0].set_title('Bronze: Enum de Lotação (bruto)')

# Silver: padronizado
sv_atv = sv_gps[sv_gps['ativo']]
cnt = sv_atv['lotacao'].value_counts()
colors = [CLR_LOT.get(k, '#888') for k in cnt.index]
axes[1].bar(cnt.index, cnt.values, color=colors, alpha=0.85, edgecolor='none')
axes[1].set_title('Silver: Lotação Padronizada (veículos ativos)')

plt.tight_layout()
plt.show()

print(f"\n  Veículos ativos na Silver:         {sv_gps['ativo'].sum():,}")
print(f"  Fora de serviço (coord 0,0 rem.): {(~sv_gps['ativo']).sum():,}")
print(f"  🔒 LGPD: motorista_id → motorista_hash (SHA-256)")
if 'motorista_hash' in sv_gps.columns:
    print(f"     Exemplo hash: {sv_gps['motorista_hash'].iloc[0]}")


In [ ]:
# ─── Ouvidoria: Geocodificação e PII ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#070b14')
fig.suptitle('Ouvidoria — Silver: Geocodificação e Status', color=CYAN, fontweight='bold')

# Mapa de pontos geocodificados
df_geo = sv_ouv.dropna(subset=['lat', 'lon'])
CLR_CAT = {'INFRAESTRUTURA': CYAN, 'MEIO_AMBIENTE': GREEN,
           'MOBILIDADE': ORANGE, 'SEGURANCA': PINK, 'OUTROS': '#607090'}
for cat, grp in df_geo.groupby('categoria'):
    axes[0].scatter(grp['lon'], grp['lat'],
                    color=CLR_CAT.get(cat, '#888'),
                    alpha=0.6, s=15, label=cat)
axes[0].set_title('Geocodificação por Categoria')
axes[0].legend(fontsize=7, loc='lower right')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Status Silver
cnt = sv_ouv['status'].value_counts()
axes[1].bar(cnt.index, cnt.values,
            color=[GREEN, GOLD, PINK, ORANGE][:len(cnt)], alpha=0.85)
axes[1].set_title('Status das Ocorrências (Silver)')

plt.tight_layout()
plt.show()

print(f"\n  Ocorrências Silver:          {len(sv_ouv):,}")
geo_ok = sv_ouv['coord_geocodificada'].sum() if 'coord_geocodificada' in sv_ouv.columns else df_geo.__len__()
print(f"  Geocodificadas:              {geo_ok:,} ({geo_ok/len(sv_ouv):.1%})")
print("\n  🔒 LGPD: CPF e telefone removidos via regex das descrições")


In [ ]:
# ─── Score de Qualidade Silver ────────────────────────────────────────────────
print("\n" + "="*65)
print("  QUALIDADE DE DADOS — SILVER (SLA: ≥ 95%)")
print("="*65)

checks = [
    ("Tráfego — timestamps UTC-aware",
     sv_tr['timestamp_utc'].notna().mean() * 100, 100),
    ("Qualidade Ar — nulos MP2.5",
     (1 - sv_ar['mp25'].isna().mean()) * 100, 95),
    ("Qualidade Ar — IQAr calculado",
     sv_ar['iqar'].notna().mean() * 100, 100),
    ("GPS — coordenadas válidas",
     sv_gps[sv_gps['ativo']]['lat'].notna().mean() * 100, 99),
    ("GPS — lotação padronizada",
     sv_gps['lotacao'].isin(['VAZIA','MEIA','CHEIA','LOTADA','DESCONHECIDA']).mean() * 100, 100),
    ("Ouvidoria — sem PII exposto",
     (1 - sv_ouv.get('texto_original', pd.Series([''])).str.contains(r'\d{3}\.\d{3}\.\d{3}-\d{2}', na=False).mean()) * 100, 100),
]

all_pass = True
for nome, valor, sla in checks:
    status = "PASS ✅" if valor >= sla else "FAIL ❌"
    bar = "█" * int(valor // 5) + "░" * (20 - int(valor // 5))
    print(f"  {status}  {nome:<40} [{bar}] {valor:.1f}%  (SLA:{sla}%)")
    if valor < sla:
        all_pass = False

print("="*65)
print(f"  Resultado: {'✅ TODOS OS CHECKS PASSARAM' if all_pass else '❌ FALHAS DETECTADAS'}")
